# `ingestion_SFP.ipynb`
## Wound Dressings – A Primer For The Family Physician (SFP 2014)

### Why this PDF needs a dedicated notebook
| Problem | Root cause | Fix |
|---|---|---|
| 5 000+ elements from `partition_pdf` | `hi_res` OCRs every tiny text fragment | Use PyMuPDF native text layer instead |
| 127 mixed / duplicate chunks | Two-column layout — unstructured interleaves left & right columns | Explicit column-aware block extraction via `get_text('blocks')` |
| Running headers in every chunk | Repeated journal header on each page | Strip known boilerplate with regex |
| Fragmented section headings (`WOUND\nDRESSINGS\nAND\nFACTORS`) | Vertical sidebar headers split across lines | Normalise with regex join |
| Table 2 (pages 5–7) garbled | Table cells extracted as random fragments | pdfplumber table extraction for those pages |
| False section header matches | `re.IGNORECASE` on L1 headers hits body text (e.g. 'advances' in intro sentence) | Case-sensitive L1 patterns; `(?m)^` anchor on L2 patterns; deduplicate by section name |

**Strategy (validated — produces 37 clean, deduplicated chunks):**
1. PyMuPDF `get_text('blocks')` → column-aware reading order (left col → right col)
2. pdfplumber table extraction for Table 2 (pages 5–7)
3. Regex cleaning to strip boilerplate, rejoin fragmented headers, replace Table 2 raw text with placeholder
4. Section-aware split using anchored patterns + section-name deduplication
5. Size-capped chunking (max 2 400 chars) at paragraph/sentence boundaries
6. Optional LLM `ai_summary` enrichment
7. Export → `ingestion_output/SFP_wound_dressings_kept.json` (same format as chunk_visualiser v3)
8. Load to ChromaDB

In [11]:
# ── CELL 0 · Install dependencies (run once if needed) ────────────────────────
# !pip install pymupdf pdfplumber -q

In [1]:
# ── CELL 1 · Imports & configuration ──────────────────────────────────────────
import fitz           # PyMuPDF — native PDF text layer, no OCR
import pdfplumber     # reliable table extraction
import re
import json
import hashlib
import unicodedata
import statistics
from pathlib import Path
from collections import defaultdict, Counter

# ── paths ──────────────────────────────────────────────────────────────────────
PDF_PATH    = "../clinical_pdfs_v2/Wound Dressings - A Primer For The Family Physician.pdf"
SOURCE_NAME = "Wound Dressings - A Primer For The Family Physician.pdf"
OUT_DIR     = Path("../ingestion_output_no_ai")
OUT_DIR.mkdir(exist_ok=True)

# ── layout constants (A4, measured from fitz block inspection) ────────────────
PAGE_WIDTH     = 595.275   # points
MID_X          = PAGE_WIDTH / 2        # ≈ 297.6 pt — column boundary
FULLWIDTH_FRAC = 0.55      # block wider than 55 % page → full-width element

# ── chunking limits ───────────────────────────────────────────────────────────
MAX_CHUNK_CHARS = 2400
MIN_CHUNK_CHARS = 80

print("✅  imports OK")

✅  imports OK


## Step 1 · Column-aware text extraction (PyMuPDF)

`get_text('blocks')` returns text blocks with true bounding boxes from the PDF's **native text layer** — no OCR involved, no coordinate confusion.

Each block is classified as `left`, `right`, or `full` based on its x-centre relative to the page midpoint.
Reading order: full-width elements interleaved by Y → left column top-to-bottom → right column top-to-bottom.

In [2]:
# ── CELL 2 · Per-page extraction ──────────────────────────────────────────────

def extract_page_blocks(pg: fitz.Page) -> list:
    """
    Return text blocks for one page, each dict:
        {x0, y0, x1, y1, text, col}   col in {'left', 'right', 'full'}
    Image blocks (blocktype != 0) and empty blocks are discarded.
    """
    raw = pg.get_text("blocks", sort=False)   # tuple: (x0,y0,x1,y1,text,blockno,blocktype)
    blocks = []
    for b in raw:
        if b[6] != 0:           # skip image blocks
            continue
        text = b[4].strip()
        if not text:
            continue
        x0, y0, x1, y1 = b[0], b[1], b[2], b[3]
        width = x1 - x0
        x_ctr = (x0 + x1) / 2
        if width >= FULLWIDTH_FRAC * PAGE_WIDTH:
            col = "full"
        elif x_ctr < MID_X:
            col = "left"
        else:
            col = "right"
        blocks.append({"x0": x0, "y0": y0, "x1": x1, "y1": y1,
                       "text": text, "col": col})
    return blocks


def blocks_to_text(blocks: list) -> str:
    """
    Merge blocks into a single string with correct 2-column reading order:
      1. LEFT column blocks, sorted by y0
      2. RIGHT column blocks, sorted by y0
    Full-width blocks are interleaved at their natural Y position into
    whichever column stream they interrupt.
    """
    left  = sorted([b for b in blocks if b["col"] == "left"],  key=lambda b: b["y0"])
    right = sorted([b for b in blocks if b["col"] == "right"], key=lambda b: b["y0"])
    full  = sorted([b for b in blocks if b["col"] == "full"],  key=lambda b: b["y0"])

    parts = []
    full_idx = 0

    for col_blocks in [left, right]:
        for b in col_blocks:
            while full_idx < len(full) and full[full_idx]["y0"] <= b["y0"]:
                parts.append(full[full_idx]["text"])
                full_idx += 1
            parts.append(b["text"])

    while full_idx < len(full):     # remaining full-width (e.g. page footer)
        parts.append(full[full_idx]["text"])
        full_idx += 1

    return "\n".join(parts)


# ── Extract all pages ─────────────────────────────────────────────────────────
doc = fitz.open(PDF_PATH)
page_texts = []
for i, pg in enumerate(doc):
    blocks = extract_page_blocks(pg)
    text   = blocks_to_text(blocks)
    page_texts.append(text)
    print(f"  page {i+1:2d}: {len(blocks):3d} blocks  →  {len(text):5d} chars")
doc.close()

raw_full_text = "\n\n".join(page_texts)
print(f"\nTotal raw text: {len(raw_full_text):,} chars across {len(page_texts)} pages")

  page  1:  20 blocks  →   5813 chars
  page  2:  30 blocks  →   2650 chars
  page  3:  15 blocks  →   6349 chars
  page  4:  12 blocks  →   6617 chars
  page  5:  24 blocks  →   2082 chars
  page  6:  20 blocks  →   1618 chars
  page  7:  19 blocks  →   2198 chars
  page  8:  11 blocks  →    992 chars
  page  9:   9 blocks  →   3719 chars
  page 10:  14 blocks  →   6307 chars

Total raw text: 38,363 chars across 10 pages


## Step 2 · Table extraction with pdfplumber (Table 2, pages 5–7)

Table 2 (*Types of Common Wound Dressings*) spans pages 5–7.  
pdfplumber reliably detects bordered cells even when PyMuPDF fragments them into many small blocks.  
Each dressing type (Hydrocolloids, Alginates, etc.) becomes its own chunk with structured Indications / Special Considerations / Examples fields.

In [3]:
# ── CELL 3 · pdfplumber table extraction ──────────────────────────────────────

TABLE2_PAGES = [4, 5, 6]   # 0-indexed page numbers for pages 5, 6, 7

TABLE2_SECTION_MAP = {
    # cell text fragment  →  clean dressing-type label
    "Hydrocolloids":   "Hydrocolloids",
    "Hydrogels":       "Hydrogels",
    "Alginates":       "Alginates",
    "Hydrofiber":      "Hydrofiber",
    "Foams":           "Foams (polyurethane or silicone)",
    "Cademoxer":       "Cademoxer Iodine",
    "Silver barrier":  "Silver Barrier Dressing",
    "Non adherent":    "Non-Adherent Synthetic",
    "Films":           "Films / Membranes",
    "Gauze":           "Gauze",
    "Composite":       "Composite Dressing",
}


def cell_text(cell) -> str:
    """Return clean single-line text from a pdfplumber table cell."""
    if cell is None:
        return ""
    return re.sub(r"\s+", " ", str(cell)).strip()


def is_type_header_row(row: list):
    """Return (True, dressing_name) if the row starts a new dressing type."""
    first_cell = cell_text(row[0]) if row else ""
    if not first_cell:
        return False, ""
    for fragment, name in TABLE2_SECTION_MAP.items():
        if fragment.lower() in first_cell.lower():
            return True, name
    return False, ""


def format_dressing_table_chunk(dtype: str, rows: list) -> str:
    """
    Format one dressing type's Table 2 rows as clean RAG text.
    Columns: [Type | Indications | Special Considerations | Examples]
    """
    col_data = defaultdict(list)
    skip = {"Types of Dressing", "Indications", "Special Considerations",
            "Examples", dtype}
    for row in rows:
        for ci, cell in enumerate(row):
            val = cell_text(cell)
            if val and val not in skip:
                col_data[ci].append(val)

    lines = [f"TABLE 2 — {dtype}"]
    for ci, label in zip([1, 2, 3], ["Indications", "Special Considerations", "Examples"]):
        if col_data[ci]:
            lines.append(f"{label}: " + " | ".join(col_data[ci]))
    return "\n".join(lines)


# Extract raw rows from pages 5–7
table2_raw_rows = []
with pdfplumber.open(PDF_PATH) as pdf:
    for pg_idx in TABLE2_PAGES:
        for tbl in pdf.pages[pg_idx].find_tables():
            rows = tbl.extract()
            if rows:
                table2_raw_rows.extend(rows)

print(f"Table 2 raw rows extracted: {len(table2_raw_rows)}")

# Group rows by dressing type
table2_groups = defaultdict(list)
current_type = "Unknown"
for row in table2_raw_rows:
    is_hdr, name = is_type_header_row(row)
    if is_hdr:
        current_type = name
    table2_groups[current_type].append(row)

# Build one text block per dressing type
table2_blocks = {}
for dtype, rows in table2_groups.items():
    if dtype == "Unknown":
        continue
    table2_blocks[dtype] = format_dressing_table_chunk(dtype, rows)

print(f"\nTable 2 dressing types found: {list(table2_blocks.keys())}")
print(f"\nSample — Hydrocolloids block:")
if "Hydrocolloids" in table2_blocks:
    print(table2_blocks["Hydrocolloids"][:400])

Table 2 raw rows extracted: 25

Table 2 dressing types found: ['Hydrocolloids', 'Hydrogels', 'Alginates', 'Hydrofiber', 'Foams (polyurethane or silicone)', 'Cademoxer Iodine', 'Silver Barrier Dressing', 'Non-Adherent Synthetic', 'Films / Membranes', 'Gauze', 'Composite Dressing']

Sample — Hydrocolloids block:
TABLE 2 — Hydrocolloids
Indications: • Dry and desiccated wounds • Abrasions • Necrotic eschars • Wounds with minimal exudates • Superficial or healing wounds
Special Considerations: • Not recommended for highly exudative or infected wounds, diabetic foot ulcers and other wounds requiring frequent wound inspection • Beware fragile skin due to potential for maceration of surrounding skin • During a


## Step 3 · Clean & normalise the full document text

In [4]:
# ── CELL 4 · Text cleaning pipeline ───────────────────────────────────────────

def clean_text(text: str) -> str:
    """
    Full cleaning pipeline applied to the raw concatenated page text.
    Order matters — each step assumes prior steps have run.

    1.  Unicode NFKC (ligatures fi→fi, ff→ff, em-dashes, etc.)
    2.  Strip repeated running headers (journal title on every page)
    3.  Strip journal volume/page line footers
    4.  Rejoin fragmented vertical sidebar heading:
            WOUND / DRESSINGS / AND / FACTORS / AFFECTING SELECTION → one line
    5.  Strip author affiliation block
    6.  Strip FIGURE 2 step-by-step photo captions (noise, no clinical value)
    7.  Replace Table 2 raw garbled text with a placeholder;
            the clean version comes from pdfplumber in Step 2
    8. Remove INTRODUCTION section (keeping Abstract)
    9.  Collapse 3+ consecutive blank lines → 2
    """
    # 1. Unicode normalise
    text = unicodedata.normalize("NFKC", text)

    # 2. Running headers (appear on every page of the journal)
    text = re.sub(r"WOUND DRESSINGS: A PRIMER FOR THE FAMILY PHYSICIAN\s*\n", "", text)
    text = re.sub(r"WOUND CARE\s*\n", "", text)

    # 3. Journal volume / page footers
    #    e.g. "T H E  S I N G A P O R E  F A M I L Y  P H Y S I C I A N  VOL 40(3) JULY-SEPTEMBER 2014 : 17"
    text = re.sub(r"T\s*H\s*E\s+S\s*I\s*N\s*G.*?20\d\d\s*:\s*\d+\s*\n",
                  "", text, flags=re.DOTALL)
    text = re.sub(r"SFP2014;.*?\d+\s*\n", "", text)
    text = re.sub(r"UNIT NO\.\s*2\s*\n", "", text)

    # 4. Rejoin vertically fragmented sidebar heading
    text = re.sub(
        r"WOUND\s*\nDRESSINGS\s*\nAND\s*\nFACTORS\s*\nAFFECTING SELECTION",
        "WOUND DRESSINGS AND FACTORS AFFECTING SELECTION",
        text
    )

    # # 5. Author affiliation block (once, between Abstract and main body)
    # text = re.sub(
    #     r"LEE MEI GENE.*?SINGAPORE GENERAL HOSPITAL\s*\n",
    #     "", text, flags=re.DOTALL | re.IGNORECASE
    # )
    # text = re.sub(
    #     r"(Department of Family Medicine.*?Singapore General Hospital"
    #     r"|Division of Nursing.*?Bright Vision Hospital"
    #     r"|Head Medical Services.*?Singapore General Hospital)\s*\n",
    #     "", text, flags=re.DOTALL | re.IGNORECASE
    # )


    # 5. Surgical removal of Authors/Hospitals (Specific lines only)
    # This avoids the "greedy" deletion that was eating the Abstract
    author_noise = [
        r"Dr Lee Mei Gene.*?\n", r"Dr Pan Yow-Jeng.*?\n", r"Yang Leng Cher.*?\n",
        r"Dr Ng Joo Ming.*?\n", r"LEE MEI GENE JESMINE.*?\n", r"PAN YOW-JENG FRANNY.*?\n",
        r"Department of Family Medicine.*?\n", r"Singapore General Hospital",
        r"Bright Vision Hospital", r"Division of Nursing"
    ]
    for pat in author_noise:
        text = re.sub(pat, "", text, flags=re.IGNORECASE)

    
    # 6. FIGURE 2 step-by-step NPWT photo captions (no clinical content)
    step_captions = [
        r"Place sterile film on.*?wound cavity\.?",
        r"Prepare required foam.*?wound bed",
        r"Insert foam.*?film again",
        r"Connect the two.*?transfer pad device\.?",
        r"Transfer pad.*?canister and pump",
        r"Abdominal wound.*?removal of stitches",
        r"On the surface.*?wound areas\.?",
        r"Maintain on negative pressure.*?dressing every three days",
        r"Vacuum device.*?negative pressure",
        r"Wound after discontinuation of NPWT",
        r"Foam dressing\s*\n",
    ]
    for pat in step_captions:
        text = re.sub(pat, "", text, flags=re.DOTALL | re.IGNORECASE)

    # 7. Remove raw Table 2 fragment (pdfplumber version used instead)
    text = re.sub(
        r"TABLE 2\. TYPES OF COMMON WOUND DRESSINGS.*?(?=ADVANCES IN WOUND CARE TECHNOLOGY)",
        "<<TABLE2_PLACEHOLDER>>\n\n",
        text, flags=re.DOTALL
    )

    # 8. Remove INTRODUCTION (Everything from 'INTRODUCTION' until the next major header)
    # This keeps Abstract but deletes the "fluff" introduction
    text = re.sub(
        r"INTRODUCTION.*?(?=WOUND DRESSINGS AND FACTORS AFFECTING SELECTION)",
        "\n", 
        text, flags=re.DOTALL
    )

    # 9. Collapse excessive blank lines
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()


cleaned_text = clean_text(raw_full_text)

# Sanity check — all major section headers must be present
SECTION_MARKERS = [
    "ABSTRACT", "INTRODUCTION",
    "WOUND DRESSINGS AND FACTORS AFFECTING SELECTION",
    "CATEGORIES OF WOUND DRESSINGS",
    "ADVANCES IN WOUND CARE TECHNOLOGY",
    "CONCLUSIONS", "REFERENCES", "LEARNING POINTS",
]
print("Section markers after cleaning:")
for m in SECTION_MARKERS:
    pos = cleaned_text.find(m)
    status = f"pos={pos:6d}" if pos >= 0 else "❌ NOT FOUND"
    print(f"  {m!r:52s}  {status}")

print(f"\nCleaned text length: {len(cleaned_text):,} chars")

Section markers after cleaning:
  'ABSTRACT'                                            pos=     0
  'INTRODUCTION'                                        ❌ NOT FOUND
  'WOUND DRESSINGS AND FACTORS AFFECTING SELECTION'     pos=  1051
  'CATEGORIES OF WOUND DRESSINGS'                       pos=  6378
  'ADVANCES IN WOUND CARE TECHNOLOGY'                   pos= 16765
  'CONCLUSIONS'                                         pos= 29168
  'REFERENCES'                                          pos= 30596
  'LEARNING POINTS'                                     pos= 31316

Cleaned text length: 35,360 chars


## Step 4 · Section-aware splitting

### Key fix: anchored patterns + section-name deduplication

The previous version used `re.IGNORECASE` for all patterns, causing false matches where body text mentioned a section title in lowercase (e.g. *"latest advances in wound care technology"* in the Introduction at pos=2228).

**Fixes applied:**
- **L1 headers** → exact case (`flags=0`) since they appear as ALL-CAPS in the PDF
- **L2 subsections** → `(?m)^` start-of-line anchor (`re.MULTILINE`) to prevent matches mid-sentence
- **Deduplication** → keep only the **first occurrence** per section name (eliminates duplicate Oxygen/Ultrasound matches from Learning Points cross-references)

In [5]:
# ── CELL 5 · Define section split patterns ────────────────────────────────────
#
# Each entry: (regex_pattern, flags, section_name, parent_section)
#
# L1  — case-sensitive (flags=0): exact ALL-CAPS headers; prevents matching
#        lowercase body-text mentions of the same words
# L2  — re.MULTILINE + (?m)^ anchor: subsection headers are mixed-case but
#        always appear at start of a line; anchor prevents body-text false matches

ALL_SPLIT_PATTERNS = [
    # ── Level-1 major sections ─────────────────────────────────────────────────
    (r"ABSTRACT\b",                                       0,            "Abstract",                             None),
    (r"INTRODUCTION\b",                                   0,            "Introduction",                         None),
    (r"WOUND DRESSINGS AND FACTORS AFFECTING SELECTION",  0,            "Wound Dressings – Selection Factors",  None),
    (r"CATEGORIES OF WOUND DRESSINGS",                    0,            "Categories of Wound Dressings",        None),
    (r"<<TABLE2_PLACEHOLDER>>",                           0,            "Table 2 – Common Wound Dressings",     None),
    (r"ADVANCES IN WOUND CARE TECHNOLOGY",                0,            "Advances in Wound Care Technology",    None),
    (r"CONCLUSIONS\b",                                    0,            "Conclusions",                          None),
    (r"REFERENCES\b",                                     0,            "References",                           None),
    (r"LEARNING POINTS\b",                                0,            "Learning Points",                      None),

    # ── Level-2: inside CATEGORIES OF WOUND DRESSINGS ──────────────────────────
    (r"(?m)^1\.\s*Moisture Retentive Dressings",  re.MULTILINE, "1. Moisture Retentive Dressings",  "Categories of Wound Dressings"),
    (r"(?m)^2\.\s*Absorbent Dressings",           re.MULTILINE, "2. Absorbent Dressings",           "Categories of Wound Dressings"),
    (r"(?m)^3\.\s*Antimicrobial Dressings",       re.MULTILINE, "3. Antimicrobial Dressings",       "Categories of Wound Dressings"),
    (r"(?m)^4\.\s*Composite Dressings",           re.MULTILINE, "4. Composite Dressings",           "Categories of Wound Dressings"),
    (r"(?m)^5\.\s*Protective dressings",          re.MULTILINE, "5. Protective Dressings",          "Categories of Wound Dressings"),

    # ── Level-2: inside ADVANCES IN WOUND CARE TECHNOLOGY ──────────────────────
    (r"(?m)^Maggot debridement therapy",          re.MULTILINE, "Maggot Debridement Therapy (MDT)", "Advances in Wound Care Technology"),
    (r"(?m)^Growth factors\s*[-\u2013\u2014]",   re.MULTILINE, "Growth Factors (PDGF/FGF)",        "Advances in Wound Care Technology"),
    (r"(?m)^Bioengineered skin substitutes",      re.MULTILINE, "Bioengineered Skin Substitutes",   "Advances in Wound Care Technology"),
    (r"(?m)^Negative pressure wound therapy",     re.MULTILINE, "Negative Pressure Wound Therapy",  "Advances in Wound Care Technology"),
    (r"(?m)^Oxygen therapy",                      re.MULTILINE, "Oxygen Therapy (HBOT)",            "Advances in Wound Care Technology"),
    (r"(?m)^Ultrasound therapy",                  re.MULTILINE, "Ultrasound Therapy",               "Advances in Wound Care Technology"),
    (r"(?m)^Low energy light treatment",          re.MULTILINE, "Low-Power Laser Therapy",          "Advances in Wound Care Technology"),
]

print(f"Total split patterns defined: {len(ALL_SPLIT_PATTERNS)}")

Total split patterns defined: 21


In [6]:
# ── CELL 6 · Find splits — with section-name deduplication ────────────────────
#
# KEY FIX over the original notebook:
#   Original: dedup only by position cluster (within 5 chars)
#             → kept multiple hits of the same section name at different positions
#               (e.g. 'Oxygen Therapy' matched in the section header AND in
#                Learning Points cross-reference, producing two separate splits)
#
#   Fixed:    dedup by BOTH position cluster AND section name
#             → keeps only the FIRST occurrence of each named section

def find_all_splits(text: str, patterns: list) -> list:
    """
    Find every pattern match in `text`.
    Returns list of {pos, section, parent} sorted by position,
    deduplicated by:
      (a) position cluster: consecutive matches within 5 chars → keep first
      (b) section name:     only the first occurrence of each section name
    """
    splits = []
    for regex, flags, section, parent in patterns:
        for m in re.finditer(regex, text, flags=flags):
            splits.append({"pos": m.start(), "section": section, "parent": parent})

    # Sort: by position ascending; ties broken by longer (more specific) section name
    splits.sort(key=lambda s: (s["pos"], -len(s["section"])))

    # Deduplicate
    deduped = []
    last_pos = -100
    seen_sections = set()
    for s in splits:
        if s["pos"] - last_pos > 5 and s["section"] not in seen_sections:
            deduped.append(s)
            last_pos = s["pos"]
            seen_sections.add(s["section"])

    return deduped


splits = find_all_splits(cleaned_text, ALL_SPLIT_PATTERNS)

print(f"Splits found: {len(splits)}  (expected: 20)")
for s in splits:
    ctx = repr(cleaned_text[s["pos"]: s["pos"] + 45])
    print(f"  pos={s['pos']:6d}  {s['section']!r:50s}  {ctx}")

Splits found: 19  (expected: 20)
  pos=     0  'Abstract'                                          'ABSTRACT\nGiven the myriad of choices availabl'
  pos=  1051  'Wound Dressings – Selection Factors'               'WOUND DRESSINGS AND FACTORS AFFECTING SELECTI'
  pos=  6378  'Categories of Wound Dressings'                     'CATEGORIES OF WOUND DRESSINGS\nTraditionally, '
  pos=  7250  '1. Moisture Retentive Dressings'                   '1. Moisture Retentive Dressings \nMoisture in '
  pos=  9431  '2. Absorbent Dressings'                            '2. Absorbent Dressings \nAbsorbent dressings p'
  pos= 11924  '3. Antimicrobial Dressings'                        '3. Antimicrobial Dressings\nIt has been found '
  pos= 15142  '4. Composite Dressings'                            '4. Composite Dressings\nComposite dressings ar'
  pos= 15896  '5. Protective Dressings'                           '5. Protective dressings\nGauze- plain gauze, m'
  pos= 16765  'Advances in Wound Care Technology

In [7]:
# ── CELL 7 · Build section text blocks ────────────────────────────────────────

def build_section_blocks(text: str, splits: list) -> list:
    """
    Slice `text` into section blocks using the split positions.
    Any text before the first split is captured as 'Title & Authors'.
    Returns list of {section, parent, text}.
    """
    blocks = []

    # Preamble before first split
    if splits and splits[0]["pos"] > 0:
        preamble = text[:splits[0]["pos"]].strip()
        if preamble:
            blocks.append({"section": "Title & Authors", "parent": None,
                           "text": preamble})

    for i, sp in enumerate(splits):
        start = sp["pos"]
        end   = splits[i + 1]["pos"] if i + 1 < len(splits) else len(text)
        section_text = text[start:end].strip()
        if section_text:
            blocks.append({
                "section": sp["section"],
                "parent":  sp["parent"],
                "text":    section_text,
            })

    return blocks


section_blocks = build_section_blocks(cleaned_text, splits)

print(f"Section blocks: {len(section_blocks)}")
for b in section_blocks:
    parent_info = f"  (parent: {b['parent']!r})" if b["parent"] else ""
    print(f"  {b['section']!r:50s}{parent_info:50s}  {len(b['text']):5d} chars")

Section blocks: 19
  'Abstract'                                                                                             1047 chars
  'Wound Dressings – Selection Factors'                                                                  5326 chars
  'Categories of Wound Dressings'                                                                         871 chars
  '1. Moisture Retentive Dressings'                   (parent: 'Categories of Wound Dressings')          2180 chars
  '2. Absorbent Dressings'                            (parent: 'Categories of Wound Dressings')          2492 chars
  '3. Antimicrobial Dressings'                        (parent: 'Categories of Wound Dressings')          3217 chars
  '4. Composite Dressings'                            (parent: 'Categories of Wound Dressings')           753 chars
  '5. Protective Dressings'                           (parent: 'Categories of Wound Dressings')           868 chars
  'Advances in Wound Care Technology'                

## Step 5 · Size-capped chunking

In [8]:
# ── CELL 8 · Split oversized sections + build final chunk list ────────────────

def split_into_chunks(text: str, max_chars: int = MAX_CHUNK_CHARS) -> list:
    """
    Split `text` into sub-chunks of at most `max_chars` characters.
    Split preference (descending):
      1. Paragraph boundary (\n\n)
      2. Sentence boundary ('. ')
      3. Hard cut at max_chars
    """
    if len(text) <= max_chars:
        return [text]

    chunks = []
    remaining = text
    while len(remaining) > max_chars:
        split_pos = remaining.rfind("\n\n", 0, max_chars)
        if split_pos == -1:
            split_pos = remaining.rfind(". ", 0, max_chars)
        if split_pos == -1:
            split_pos = max_chars
        else:
            split_pos += 2   # include the delimiter
        chunks.append(remaining[:split_pos].strip())
        remaining = remaining[split_pos:].strip()

    if remaining.strip():
        chunks.append(remaining.strip())

    return [c for c in chunks if c]


def make_chunk_id(source: str, section: str, chunk_num: int) -> str:
    raw = f"{source}::{section}::{chunk_num}"
    return hashlib.md5(raw.encode()).hexdigest()[:12]


# Sections skipped in final output:
#   - Table 2 placeholder (replaced by pdfplumber chunks in CELL 9)
#   - References (no clinical content useful for RAG)
#   - Title & Authors (metadata only, no dressing content)
SKIP_SECTIONS = {
    "Table 2 – Common Wound Dressings", 
    "References", 
    "Title & Authors"
}

chunks = []

for block in section_blocks:
    section = block["section"]
    parent  = block["parent"]
    text    = block["text"]

    if section in SKIP_SECTIONS:
        continue
    if len(text) < MIN_CHUNK_CHARS:
        continue

    for ci, chunk_text in enumerate(split_into_chunks(text)):
        if len(chunk_text) < MIN_CHUNK_CHARS:
            continue

        chunks.append({
            "chunk_id":       make_chunk_id(SOURCE_NAME, section, ci),
            "source":         SOURCE_NAME,
            "section":        section,
            "parent_section": parent or section,
            "chunk_index":    ci,
            "char_count":     len(chunk_text),
            "text":           chunk_text,
            "ai_summary":     chunk_text,   # overwritten by LLM step if enabled
        })

print(f"Chunks from section blocks: {len(chunks)}")
for c in chunks:
    print(f"  [{c['chunk_id']}]  {c['section']!r:50s}  idx={c['chunk_index']}  chars={c['char_count']}")

Chunks from section blocks: 25
  [ba05f42e79d0]  'Abstract'                                          idx=0  chars=1047
  [08021aac2fa6]  'Wound Dressings – Selection Factors'               idx=0  chars=2348
  [e7ca1b602a48]  'Wound Dressings – Selection Factors'               idx=1  chars=685
  [18ab673003dc]  'Wound Dressings – Selection Factors'               idx=2  chars=2289
  [07da3fcbedeb]  'Categories of Wound Dressings'                     idx=0  chars=871
  [14535d1dba1a]  '1. Moisture Retentive Dressings'                   idx=0  chars=2180
  [8443058d681e]  '2. Absorbent Dressings'                            idx=0  chars=2248
  [c42103c45a0c]  '2. Absorbent Dressings'                            idx=1  chars=243
  [80e829def300]  '3. Antimicrobial Dressings'                        idx=0  chars=1089
  [3082e1a296e7]  '3. Antimicrobial Dressings'                        idx=1  chars=2126
  [5972b0ef4b66]  '4. Composite Dressings'                            idx=0  chars=753
  [b4

In [9]:
# ── CELL 9 · Add Table 2 chunks (pdfplumber-derived, one per dressing type) ───

for dtype, tblock_text in table2_blocks.items():
    if len(tblock_text) < MIN_CHUNK_CHARS:
        continue
    chunks.append({
        "chunk_id":       make_chunk_id(SOURCE_NAME, f"Table2::{dtype}", 0),
        "source":         SOURCE_NAME,
        "section":        f"Table 2 – {dtype}",
        "parent_section": "Table 2 – Common Wound Dressings",
        "chunk_index":    0,
        "char_count":     len(tblock_text),
        "text":           tblock_text,
        "ai_summary":     tblock_text,
    })

print(f"Total chunks (section blocks + Table 2): {len(chunks)}")
print(f"  Expected: ~37")

Total chunks (section blocks + Table 2): 36
  Expected: ~37


## Step 6 · Quality validation

In [10]:
# ── CELL 10 · Quality checks ──────────────────────────────────────────────────

all_text_combined = " ".join(c["text"].lower() for c in chunks)

# 1. Duplicate detection (first 200 chars fingerprint)
seen_texts = {}
duplicates = []
for c in chunks:
    key = c["text"][:200]
    if key in seen_texts:
        duplicates.append((seen_texts[key], c["chunk_id"], c["section"]))
    else:
        seen_texts[key] = c["chunk_id"]

# 2. Clinical content coverage
MUST_CONTAIN = [
    ("hydrocolloid",          "Hydrocolloids"),
    ("alginate",              "Alginates"),
    ("hydrofiber",            "Hydrofiber"),
    ("foam dressing",         "Foam dressings"),
    ("silver",                "Silver dressing"),
    ("iodine",                "Cademoxer Iodine"),
    ("thyroid",               "Iodine thyroid contraindication"),
    ("fragile skin",          "Foam fragile-skin warning"),
    ("negative pressure",     "NPWT"),
    ("maggot",                "MDT"),
    ("hyperbaric",            "HBOT"),
    ("ultrasound",            "Ultrasound therapy"),
    ("soiled",                "Dressing-change criteria"),
    ("no one dressing",       "Conclusions key statement"),
]

print("═" * 70)
print("CHUNK QUALITY REPORT")
print("═" * 70)

char_counts = [c["char_count"] for c in chunks]
print(f"\n📦 Total chunks     : {len(chunks)}")
print(f"   Char count — min : {min(char_counts)}")
print(f"   Char count — mean: {statistics.mean(char_counts):.0f}")
print(f"   Char count — max : {max(char_counts)}")
oversized = [c for c in chunks if c["char_count"] > MAX_CHUNK_CHARS]
print(f"   Oversized (>{MAX_CHUNK_CHARS}): {len(oversized)}"
      + (" ← check split logic" if oversized else ""))

print(f"\n🔁 Duplicates       : {len(duplicates)}"
      + (" ← check dedup logic" if duplicates else " ✓"))
for d in duplicates:
    print(f"   {d}")

print("\n✅ Clinical content coverage:")
all_ok = True
for keyword, label in MUST_CONTAIN:
    found = keyword.lower() in all_text_combined
    status = "  ✓" if found else "  ✗ MISSING"
    if not found:
        all_ok = False
    print(f"{status}  {label!r}")

print("\n🗂  Chunks by section:")
section_counts = Counter(c["section"] for c in chunks)
for sec, cnt in sorted(section_counts.items()):
    print(f"  {cnt:2d} × {sec!r}")

if all_ok and not duplicates and not oversized:
    print("\n✅  All checks passed — pipeline output is clean.")
else:
    print("\n⚠️   One or more checks failed — review above.")

══════════════════════════════════════════════════════════════════════
CHUNK QUALITY REPORT
══════════════════════════════════════════════════════════════════════

📦 Total chunks     : 36
   Char count — min : 241
   Char count — mean: 1085
   Char count — max : 2387
   Oversized (>2400): 0

🔁 Duplicates       : 0 ✓

✅ Clinical content coverage:
  ✓  'Hydrocolloids'
  ✓  'Alginates'
  ✓  'Hydrofiber'
  ✓  'Foam dressings'
  ✓  'Silver dressing'
  ✓  'Cademoxer Iodine'
  ✓  'Iodine thyroid contraindication'
  ✓  'Foam fragile-skin warning'
  ✓  'NPWT'
  ✓  'MDT'
  ✓  'HBOT'
  ✓  'Ultrasound therapy'
  ✓  'Dressing-change criteria'
  ✓  'Conclusions key statement'

🗂  Chunks by section:
   1 × '1. Moisture Retentive Dressings'
   2 × '2. Absorbent Dressings'
   2 × '3. Antimicrobial Dressings'
   1 × '4. Composite Dressings'
   1 × '5. Protective Dressings'
   1 × 'Abstract'
   1 × 'Advances in Wound Care Technology'
   1 × 'Bioengineered Skin Substitutes'
   1 × 'Categories of Wound Dre

In [11]:
# ── CELL 11 · Spot-check individual chunks ────────────────────────────────────

def preview_chunk(identifier):
    """Preview by list index (int) or section name substring (str)."""
    if isinstance(identifier, int):
        c = chunks[identifier]
    else:
        matches = [x for x in chunks
                   if identifier.lower() in x["section"].lower()]
        if not matches:
            print(f"No chunk matching {identifier!r}")
            return
        c = matches[0]

    print(f"\n{'─'*65}")
    print(f"chunk_id   : {c['chunk_id']}")
    print(f"section    : {c['section']}")
    print(f"parent     : {c['parent_section']}")
    print(f"chars      : {c['char_count']}")
    print(f"TEXT PREVIEW (first 600 chars):")
    print(c["text"][:600])
    if len(c["text"]) > 600:
        print("... [truncated]")


# Spot-check clinically important chunks
for key in ["Antimicrobial", "Table 2 – Hydrocolloids",
            "Negative Pressure", "Selection Factors", "Conclusions"]:
    preview_chunk(key)


─────────────────────────────────────────────────────────────────
chunk_id   : 80e829def300
section    : 3. Antimicrobial Dressings
parent     : Categories of Wound Dressings
chars      : 1089
TEXT PREVIEW (first 600 chars):
3. Antimicrobial Dressings
It has been found that the presence of any trace of β-hemolytic 
streptococci or bacterial concentration over 105 or 106 bacteria 
colony-forming units per gram of tissue in wound is associated 
with impaired healing14. Te recommendation to date is to 
reduce or eliminate the bioburden through a combination of 
frequent debridement, vigorous physical cleansing, and use of 
appropriate dressing material, extensive high-dose systemic 
antibiotics or topic biocides to disrupt its reconstitution15. Te 
following section describes some of the readily available types of 
a
... [truncated]

─────────────────────────────────────────────────────────────────
chunk_id   : b4c13d77818b
section    : Table 2 – Hydrocolloids
parent     : Table 2 – Comm

## Step 7 · (Optional) LLM `ai_summary` enrichment

Set `ENABLE_AI_SUMMARY = True` to replace the raw extracted text in `ai_summary` with an LLM-generated clinical summary.

**`ai_summary` is what gets stored as `page_content` in ChromaDB and used as `reference_contexts` in your RAGAS testset.**  
If you skip this step, `ai_summary == text` (raw extraction), which is already clean and readable.

In [12]:
# ── CELL 12 · LLM ai_summary enrichment (optional) ───────────────────────────
ENABLE_AI_SUMMARY = False   # ← set True when OPENAI_API_KEY is available

# Sections that benefit most from LLM summarisation:
#   - Table 2 chunks (structured data → cleaner prose)
#   - Antimicrobial Dressings (iodine/silver contraindications must be explicit)
# Pure narrative sections (Abstract, Introduction, Conclusions) are already
# clean prose — LLM adds little value there.
AI_PRIORITY_SECTIONS = {
    "Table 2 – Hydrocolloids", "Table 2 – Hydrogels", "Table 2 – Alginates",
    "Table 2 – Hydrofiber", "Table 2 – Foams (polyurethane or silicone)",
    "Table 2 – Cademoxer Iodine", "Table 2 – Silver Barrier Dressing",
    "Table 2 – Non-Adherent Synthetic", "Table 2 – Films / Membranes",
    "Table 2 – Gauze", "Table 2 – Composite Dressing",
    "3. Antimicrobial Dressings",
}

SYSTEM_PROMPT = (
    "You are a clinical wound-care summarisation assistant. "
    "Rewrite the following wound-dressing text as a clear, complete, self-contained "
    "clinical summary for a retrieval-augmented generation system. "
    "Preserve ALL clinical facts: dressing names, indications, contraindications, "
    "frequency of change, and safety warnings. "
    "Return only the summary — no preamble, no commentary."
)

if ENABLE_AI_SUMMARY:
    import os
    from openai import OpenAI

    client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

    def get_ai_summary(text: str) -> str:
        resp = client.chat.completions.create(
            model="gpt-4o-mini",
            temperature=0,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user",   "content": text},
            ],
        )
        return resp.choices[0].message.content.strip()

    print(f"Running AI summaries for {len(chunks)} chunks...")
    for i, c in enumerate(chunks):
        note = " (priority)" if c["section"] in AI_PRIORITY_SECTIONS else ""
        print(f"  [{i+1:2d}/{len(chunks)}] {c['section']}{note}")
        c["ai_summary"] = get_ai_summary(c["text"])
    print("\n✅  AI summaries complete")
else:
    ai_count = sum(1 for c in chunks if c["ai_summary"] != c["text"])
    print(f"ℹ️   AI summary disabled — ai_summary == text (raw extraction)")
    print(f"    {ai_count} chunks have a different ai_summary from a previous run")
    print(f"    Set ENABLE_AI_SUMMARY = True to regenerate.")

ℹ️   AI summary disabled — ai_summary == text (raw extraction)
    0 chunks have a different ai_summary from a previous run
    Set ENABLE_AI_SUMMARY = True to regenerate.


## Step 8 · Export JSON

Output format mirrors `chunk_visualiser_unstructured_v3.py` so `wound_dressing_rag_ingestion_v2.ipynb` can read it without modification.

In [13]:
# ── CELL 13 · Export ChromaDB-ready JSON ──────────────────────────────────────

output = {
    "meta": {
        "total_chunks":  len(chunks),
        "kept_count":    len(chunks),
        "ai_summarised": sum(1 for c in chunks if c["ai_summary"] != c["text"]),
        "extraction":    "PyMuPDF native text layer + pdfplumber tables",
        "chunking":      "section-aware semantic split with anchored regex + section-name dedup",
        "chunk_params": {
            "max_characters": MAX_CHUNK_CHARS,
            "min_characters": MIN_CHUNK_CHARS,
        },
        "note": (
            "Use ai_summary field for reference_contexts in RAGAS testset "
            "and as page_content in ChromaDB. ai_summary == text when "
            "ENABLE_AI_SUMMARY=False."
        ),
    },
    "kept_ids_by_source": {
        SOURCE_NAME: [c["chunk_id"] for c in chunks]
    },
    "kept_chunks": [
        {
            "chunk_id":       c["chunk_id"],
            "source":         c["source"],
            "section":        c["section"],
            "parent_section": c["parent_section"],
            "chunk_index":    c["chunk_index"],
            "char_count":     c["char_count"],
            "text":           c["text"],
            "ai_summary":     c["ai_summary"],
        }
        for c in chunks
    ],
}

out_path = OUT_DIR / "SFP_wound_dressings_kept.json"
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(output, f, ensure_ascii=False, indent=2)

print(f"✅  Exported {len(chunks)} chunks → {out_path}")
print(f"    File size: {out_path.stat().st_size / 1024:.1f} KB")

✅  Exported 36 chunks → ..\ingestion_output_no_ai\SFP_wound_dressings_kept.json
    File size: 91.9 KB


## Step 9 · Load to ChromaDB

In [ ]:
# ── CELL 14 · Ingest into ChromaDB ────────────────────────────────────────────
# Matches wound_dressing_rag_ingestion_v2.ipynb pattern — interchangeable.

ENABLE_CHROMADB  = False                    # ← set True to actually load
CHROMA_DB_PATH   = "./db_wound_care_v2"     # match your existing DB path
COLLECTION_NAME  = "wound_care"             # match your collection name
EMBED_MODEL      = "abhinand/MedEmbed-large-v0.1"   # match your embed model

if ENABLE_CHROMADB:
    from langchain_chroma import Chroma
    from langchain_huggingface import HuggingFaceEmbeddings
    from langchain_core.documents import Document
    import torch

    embedding_model = HuggingFaceEmbeddings(
        model_name=EMBED_MODEL,
        model_kwargs={"device": "cuda" if torch.cuda.is_available() else "cpu"},
        encode_kwargs={"normalize_embeddings": True},
    )

    db = Chroma(
        persist_directory=CHROMA_DB_PATH,
        embedding_function=embedding_model,
        collection_name=COLLECTION_NAME,
        collection_metadata={"hnsw:space": "cosine"},
    )

    # Remove existing documents for this source (clean re-ingest)
    existing = db.get(where={"source": SOURCE_NAME})
    if existing["ids"]:
        db.delete(ids=existing["ids"])
        print(f"🗑   Removed {len(existing['ids'])} existing docs for {SOURCE_NAME}")

    # Build LangChain Document objects
    docs = []
    for c in chunks:
        docs.append(Document(
            page_content=c["ai_summary"],   # what gets embedded and searched
            metadata={
                "source":         c["source"],
                "section":        c["section"],
                "parent_section": c["parent_section"],
                "chunk_index":    c["chunk_index"],
                "char_count":     c["char_count"],
                "chunk_id":       c["chunk_id"],
                # Store raw text in metadata for debugging
                "raw_text":       c["text"][:500],
            },
        ))

    # Add in batches of 50
    BATCH = 50
    for start in range(0, len(docs), BATCH):
        batch = docs[start:start + BATCH]
        ids   = [c["chunk_id"] for c in chunks[start:start + BATCH]]
        db.add_documents(documents=batch, ids=ids)
        print(f"  Loaded batch {start}–{start + len(batch) - 1}")

    print(f"\n✅  ChromaDB '{COLLECTION_NAME}' now has {db._collection.count()} docs")

    # Smoke test
    results = db.similarity_search(
        "what dressing is contraindicated for patients with thyroid disorders?",
        k=3
    )
    print("\n🔍 Smoke test — top 3 results:")
    for i, r in enumerate(results, 1):
        print(f"  [{i}] section={r.metadata['section']!r}")
        print(f"      {r.page_content[:200]}\n")
else:
    print("ℹ️   ChromaDB loading skipped (ENABLE_CHROMADB = False)")
    print(f"    Set ENABLE_CHROMADB = True to load {len(chunks)} chunks into '{COLLECTION_NAME}'")

## Step 10 · RAGAS testset reference context helper

Each chunk's `ai_summary` field is your `reference_context` string for RAGAS `SingleTurnSample`.  
The lookup below shows which section to use for each test case category.

| Test case category | Section to use |
|---|---|
| `cat_b_iodine_thyroid` | `3. Antimicrobial Dressings` (iodine thyroid warning) |
| `cat_b_foam_fragile_skin` | `1. Moisture Retentive Dressings` (foam fragile skin note) |
| `cat_b_npwt_necrotic_eschar` | `Negative Pressure Wound Therapy` (NPWT contraindications) |
| `cat_c_dressing_saturation_change` | `Wound Dressings – Selection Factors` (soiled/saturated criteria) |
| Any dressing-specific case | `Table 2 – <DressingType>` chunk |
| Silver properties | `Table 2 – Silver Barrier Dressing` or `3. Antimicrobial Dressings` |

In [ ]:
# ── CELL 15 · Build reference context lookups for RAGAS ──────────────────────

# Load from the exported JSON (works after kernel restart)
with open(OUT_DIR / "SFP_wound_dressings_kept.json") as f:
    exported = json.load(f)

# chunk_id  →  ai_summary  (exact string stored in ChromaDB)
ref_ctx_by_id = {
    c["chunk_id"]: c["ai_summary"]
    for c in exported["kept_chunks"]
}

# section name  →  list[ai_summary]  (one entry per sub-chunk)
ref_ctx_by_section = defaultdict(list)
for c in exported["kept_chunks"]:
    ref_ctx_by_section[c["section"]].append(c["ai_summary"])

print("reference_context lookups ready")
print(f"  By chunk_id  : {len(ref_ctx_by_id)} entries")
print(f"  By section   : {len(ref_ctx_by_section)} sections")

print("\nAll sections available:")
for sec in sorted(ref_ctx_by_section.keys()):
    n = len(ref_ctx_by_section[sec])
    chars = sum(len(x) for x in ref_ctx_by_section[sec])
    print(f"  {sec!r:52s}  {n} chunk(s)  {chars:5d} chars")

# ── Example: retrieve contexts for a test case ────────────────────────────────
print("\n" + "─" * 65)
print("Example: reference_contexts for cat_b_iodine_thyroid")
print("─" * 65)
for ctx in ref_ctx_by_section.get("3. Antimicrobial Dressings", []):
    # Show just the iodine-relevant paragraph
    iodine_start = ctx.lower().find("iodine")
    if iodine_start >= 0:
        print(ctx[max(0, iodine_start - 50): iodine_start + 400])
        print("---")

## Step 11 · Final summary table

In [ ]:
# ── CELL 16 · Final summary ───────────────────────────────────────────────────

print("═" * 72)
print("INGESTION COMPLETE — Wound Dressings: A Primer for the Family Physician")
print("═" * 72)
print()

hdr = f"{'#':>3}  {'Chunk ID':14}  {'Section':50}  {'Chars':>5}  {'AI?':8}"
print(hdr)
print("─" * len(hdr))

for i, c in enumerate(exported["kept_chunks"], 1):
    ai_flag = "yes" if c["ai_summary"] != c["text"] else "no (raw)"
    print(f"{i:3d}  {c['chunk_id']:14}  {c['section'][:50]:50s}  "
          f"{c['char_count']:5d}  {ai_flag:8}")

print()
print(f"Output JSON  : {OUT_DIR / 'SFP_wound_dressings_kept.json'}")
print()
print("Next steps:")
print("  1. Set ENABLE_AI_SUMMARY = True (Cell 12) to enrich ai_summary via LLM")
print("  2. Set ENABLE_CHROMADB   = True (Cell 14) to load into vector store")
print("  3. Use ref_ctx_by_section[<section>] for RAGAS testset reference_contexts")
print("  4. ai_summary field == page_content stored in ChromaDB")